In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

TRANSFORM_RUN_ID = f"silver_synthetic_internal_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "12_silver_synthetic_internal_tables"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

bronze_synthetic_tables = [
    "bronze_suppliers",
    "bronze_supplier_aliases",
    "bronze_supplier_sku_map",
    "bronze_alternate_suppliers",
    "bronze_supplier_scorecards",
    "bronze_purchase_orders",
    "bronze_shipment_routes",
    "bronze_product_crosswalk"
]

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("NOTEBOOK_NAME:", NOTEBOOK_NAME)

table_inventory_rows = []

for table_name in bronze_synthetic_tables:
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    try:
        df = spark.table(full_table_name)
        row_count = df.count()
        status = "FOUND"

        table_inventory_rows.append({
            "table_name": table_name,
            "full_table_name": full_table_name,
            "row_count": row_count,
            "status": status
        })

        print(f"{table_name}: {row_count} rows")

    except Exception as e:
        table_inventory_rows.append({
            "table_name": table_name,
            "full_table_name": full_table_name,
            "row_count": 0,
            "status": "MISSING"
        })

        print(f"{table_name}: MISSING")


table_inventory_schema = T.StructType([
    T.StructField("table_name", T.StringType(), True),
    T.StructField("full_table_name", T.StringType(), True),
    T.StructField("row_count", T.LongType(), True),
    T.StructField("status", T.StringType(), True)
])

table_inventory_df = spark.createDataFrame(table_inventory_rows, schema=table_inventory_schema)

display(table_inventory_df.orderBy("table_name"))

missing_table_count = table_inventory_df.filter(F.col("status") == "MISSING").count()

print("Missing Bronze synthetic tables:", missing_table_count)

if missing_table_count == 0:
    print("✅ Chunk 1 PASSED: All Bronze synthetic internal tables are available.")
else:
    print("❌ Chunk 1 FAILED: Some Bronze synthetic internal tables are missing.")

In [0]:
# Notebook 12, Chunk 2: Inspect schemas for Bronze synthetic/internal tables

for table_name in bronze_synthetic_tables:
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"
    
    print("=" * 100)
    print(full_table_name)
    print("=" * 100)
    
    df = spark.table(full_table_name)
    
    print("Row count:", df.count())
    print("Columns:")
    for col_name in df.columns:
        print(f"  - {col_name}")
    
    print("Schema:")
    df.printSchema()
    
    print("Sample rows:")
    display(df.limit(5))

In [0]:
shipment_routes_df = spark.table(f"{BRONZE_SCHEMA}.bronze_shipment_routes")

print("bronze_shipment_routes row count:", shipment_routes_df.count())

print("Columns:")
for col_name in shipment_routes_df.columns:
    print(f"  - {col_name}")

shipment_routes_df.printSchema()

display(shipment_routes_df.limit(10))

In [0]:
def clean_string_col(col_name):
    return (
        F.when(F.col(col_name).isNull(), None)
        .when(F.trim(F.col(col_name)) == "", None)
        .when(F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan"), None)
        .otherwise(F.trim(F.col(col_name)))
    )


def add_silver_metadata(df):
    return (
        df
        .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
        .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("silver_created_by", F.lit(CREATED_BY))
        .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
    )


bronze_suppliers_df = spark.table(f"{BRONZE_SCHEMA}.bronze_suppliers")
bronze_supplier_aliases_df = spark.table(f"{BRONZE_SCHEMA}.bronze_supplier_aliases")
bronze_supplier_sku_map_df = spark.table(f"{BRONZE_SCHEMA}.bronze_supplier_sku_map")
bronze_alternate_suppliers_df = spark.table(f"{BRONZE_SCHEMA}.bronze_alternate_suppliers")
bronze_supplier_scorecards_df = spark.table(f"{BRONZE_SCHEMA}.bronze_supplier_scorecards")
bronze_purchase_orders_df = spark.table(f"{BRONZE_SCHEMA}.bronze_purchase_orders")
bronze_shipment_routes_df = spark.table(f"{BRONZE_SCHEMA}.bronze_shipment_routes")
bronze_product_crosswalk_df = spark.table(f"{BRONZE_SCHEMA}.bronze_product_crosswalk")


silver_suppliers_df = add_silver_metadata(
    bronze_suppliers_df
    .select(
        clean_string_col("supplier_id").alias("supplier_id"),
        clean_string_col("supplier_name").alias("supplier_name"),
        clean_string_col("parent_company").alias("parent_company"),
        clean_string_col("country").alias("country"),
        clean_string_col("region").alias("region"),
        clean_string_col("supplier_category").alias("supplier_category"),
        clean_string_col("criticality_tier").alias("criticality_tier"),
        F.col("annual_spend").cast("double").alias("annual_spend"),
        F.col("single_source_flag").cast("boolean").alias("single_source_flag"),
        F.col("default_lead_time_days").cast("int").alias("default_lead_time_days"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn("supplier_record_id", F.sha2(F.concat_ws("||", "supplier_id", "supplier_name"), 256))
)


silver_supplier_aliases_df = add_silver_metadata(
    bronze_supplier_aliases_df
    .select(
        clean_string_col("supplier_id").alias("supplier_id"),
        clean_string_col("alias_name").alias("alias_name"),
        clean_string_col("alias_type").alias("alias_type"),
        F.col("match_confidence").cast("double").alias("match_confidence"),
        clean_string_col("source").alias("source"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn("supplier_alias_record_id", F.sha2(F.concat_ws("||", "supplier_id", "alias_name", "alias_type"), 256))
)


silver_supplier_sku_map_df = add_silver_metadata(
    bronze_supplier_sku_map_df
    .select(
        clean_string_col("supplier_id").alias("supplier_id"),
        clean_string_col("sku_id").alias("sku_id"),
        F.col("dependency_percent").cast("double").alias("dependency_percent"),
        F.col("is_primary_supplier").cast("boolean").alias("is_primary_supplier"),
        F.col("alternate_supplier_available").cast("boolean").alias("alternate_supplier_available"),
        F.col("standard_lead_time_days").cast("int").alias("standard_lead_time_days"),
        clean_string_col("origin_country").alias("origin_country"),
        clean_string_col("origin_port").alias("origin_port"),
        clean_string_col("destination_dc").alias("destination_dc"),
        clean_string_col("transport_mode").alias("transport_mode"),
        F.col("minimum_order_quantity").cast("int").alias("minimum_order_quantity"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn("supplier_sku_record_id", F.sha2(F.concat_ws("||", "supplier_id", "sku_id"), 256))
)


silver_alternate_suppliers_df = add_silver_metadata(
    bronze_alternate_suppliers_df
    .select(
        clean_string_col("sku_id").alias("sku_id"),
        clean_string_col("primary_supplier_id").alias("primary_supplier_id"),
        clean_string_col("alternate_supplier_id").alias("alternate_supplier_id"),
        F.col("approved_flag").cast("boolean").alias("approved_flag"),
        clean_string_col("switching_cost_level").alias("switching_cost_level"),
        F.col("estimated_switch_days").cast("int").alias("estimated_switch_days"),
        F.col("capacity_available_pct").cast("double").alias("capacity_available_pct"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn(
        "alternate_supplier_record_id",
        F.sha2(F.concat_ws("||", "sku_id", "primary_supplier_id", "alternate_supplier_id"), 256)
    )
)


silver_supplier_scorecards_df = add_silver_metadata(
    bronze_supplier_scorecards_df
    .select(
        clean_string_col("supplier_id").alias("supplier_id"),
        clean_string_col("scorecard_month").alias("scorecard_month_raw"),
        F.col("fill_rate").cast("double").alias("fill_rate"),
        F.col("on_time_delivery_rate").cast("double").alias("on_time_delivery_rate"),
        F.col("quality_issue_rate").cast("double").alias("quality_issue_rate"),
        F.col("avg_lead_time_days").cast("double").alias("avg_lead_time_days"),
        F.col("lead_time_variance").cast("double").alias("lead_time_variance"),
        F.col("defect_rate").cast("double").alias("defect_rate"),
        F.col("is_deteriorating").cast("boolean").alias("is_deteriorating"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn(
        "scorecard_month",
        F.coalesce(
            F.to_date(F.col("scorecard_month_raw")),
            F.to_date(F.concat(F.col("scorecard_month_raw"), F.lit("-01")))
        )
    )
    .withColumn("scorecard_year", F.year(F.col("scorecard_month")))
    .withColumn("scorecard_month_number", F.month(F.col("scorecard_month")))
    .withColumn(
        "supplier_scorecard_record_id",
        F.sha2(F.concat_ws("||", "supplier_id", "scorecard_month_raw"), 256)
    )
)


silver_purchase_orders_df = add_silver_metadata(
    bronze_purchase_orders_df
    .select(
        clean_string_col("po_id").alias("po_id"),
        clean_string_col("po_line_id").alias("po_line_id"),
        clean_string_col("supplier_id").alias("supplier_id"),
        clean_string_col("sku_id").alias("sku_id"),
        F.to_date(F.col("order_date")).alias("order_date"),
        F.to_date(F.col("expected_delivery_date")).alias("expected_delivery_date"),
        F.to_date(F.col("actual_delivery_date")).alias("actual_delivery_date"),
        F.col("quantity_ordered").cast("int").alias("quantity_ordered"),
        F.col("quantity_received").cast("int").alias("quantity_received"),
        clean_string_col("status").alias("status"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn("order_date_id", F.date_format(F.col("order_date"), "yyyyMMdd").cast("int"))
    .withColumn("expected_delivery_date_id", F.date_format(F.col("expected_delivery_date"), "yyyyMMdd").cast("int"))
    .withColumn("actual_delivery_date_id", F.date_format(F.col("actual_delivery_date"), "yyyyMMdd").cast("int"))
    .withColumn("planned_lead_time_days", F.datediff(F.col("expected_delivery_date"), F.col("order_date")))
    .withColumn("actual_lead_time_days", F.datediff(F.col("actual_delivery_date"), F.col("order_date")))
    .withColumn("delivery_delay_days", F.datediff(F.col("actual_delivery_date"), F.col("expected_delivery_date")))
    .withColumn("is_po_late", F.when(F.col("delivery_delay_days") > 0, F.lit(True)).otherwise(F.lit(False)))
    .withColumn(
        "po_fill_rate",
        F.when(F.col("quantity_ordered") > 0, F.col("quantity_received") / F.col("quantity_ordered")).otherwise(None)
    )
    .withColumn("purchase_order_record_id", F.sha2(F.concat_ws("||", "po_id", "po_line_id", "supplier_id", "sku_id"), 256))
)


silver_shipment_routes_df = add_silver_metadata(
    bronze_shipment_routes_df
    .select(
        clean_string_col("route_id").alias("route_id"),
        clean_string_col("supplier_id").alias("supplier_id"),
        clean_string_col("origin_country").alias("origin_country"),
        clean_string_col("origin_port").alias("origin_port"),
        clean_string_col("destination_dc").alias("destination_dc"),
        clean_string_col("transport_mode").alias("transport_mode"),
        clean_string_col("carrier").alias("carrier"),
        F.col("standard_transit_days").cast("int").alias("standard_transit_days"),
        clean_string_col("risk_region").alias("risk_region"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn("shipment_route_record_id", F.sha2(F.concat_ws("||", "route_id", "supplier_id", "origin_port", "destination_dc"), 256))
)


silver_product_crosswalk_df = add_silver_metadata(
    bronze_product_crosswalk_df
    .select(
        clean_string_col("canonical_sku_id").alias("canonical_sku_id"),
        clean_string_col("source_system").alias("source_system"),
        clean_string_col("source_product_id").alias("source_product_id"),
        clean_string_col("source_product_name").alias("source_product_name"),
        clean_string_col("source_category").alias("source_category"),
        clean_string_col("source_department").alias("source_department"),
        F.col("confidence").cast("double").alias("confidence"),
        clean_string_col("mapping_rule").alias("mapping_rule"),
        clean_string_col("synthetic_batch_id").alias("synthetic_batch_id"),
        F.col("is_synthetic").cast("boolean").alias("is_synthetic")
    )
    .withColumn(
        "product_crosswalk_record_id",
        F.sha2(F.concat_ws("||", "canonical_sku_id", "source_system", "source_product_id"), 256)
    )
)


silver_internal_dfs = {
    "silver_suppliers": silver_suppliers_df,
    "silver_supplier_aliases": silver_supplier_aliases_df,
    "silver_supplier_sku_map": silver_supplier_sku_map_df,
    "silver_alternate_suppliers": silver_alternate_suppliers_df,
    "silver_supplier_scorecards": silver_supplier_scorecards_df,
    "silver_purchase_orders": silver_purchase_orders_df,
    "silver_shipment_routes": silver_shipment_routes_df,
    "silver_product_crosswalk": silver_product_crosswalk_df
}

for table_name, df in silver_internal_dfs.items():
    print(f"{table_name}: {df.count()} rows")

print("✅ Chunk 3 complete: Silver synthetic/internal dataframes created.")

In [0]:
# Notebook 12, Chunk 4: Validate Silver synthetic/internal DataFrames before writing

synthetic_validation_rows = []

bronze_to_silver_table_map = [
    {
        "bronze_table": "bronze_suppliers",
        "silver_table": "silver_suppliers",
        "df": silver_suppliers_df,
        "key_columns": ["supplier_id"],
        "record_id_column": "supplier_record_id"
    },
    {
        "bronze_table": "bronze_supplier_aliases",
        "silver_table": "silver_supplier_aliases",
        "df": silver_supplier_aliases_df,
        "key_columns": ["supplier_id", "alias_name"],
        "record_id_column": "supplier_alias_record_id"
    },
    {
        "bronze_table": "bronze_supplier_sku_map",
        "silver_table": "silver_supplier_sku_map",
        "df": silver_supplier_sku_map_df,
        "key_columns": ["supplier_id", "sku_id"],
        "record_id_column": "supplier_sku_record_id"
    },
    {
        "bronze_table": "bronze_alternate_suppliers",
        "silver_table": "silver_alternate_suppliers",
        "df": silver_alternate_suppliers_df,
        "key_columns": ["sku_id", "primary_supplier_id", "alternate_supplier_id"],
        "record_id_column": "alternate_supplier_record_id"
    },
    {
        "bronze_table": "bronze_supplier_scorecards",
        "silver_table": "silver_supplier_scorecards",
        "df": silver_supplier_scorecards_df,
        "key_columns": ["supplier_id", "scorecard_month"],
        "record_id_column": "supplier_scorecard_record_id"
    },
    {
        "bronze_table": "bronze_purchase_orders",
        "silver_table": "silver_purchase_orders",
        "df": silver_purchase_orders_df,
        "key_columns": ["po_id", "po_line_id", "supplier_id", "sku_id"],
        "record_id_column": "purchase_order_record_id"
    },
    {
        "bronze_table": "bronze_shipment_routes",
        "silver_table": "silver_shipment_routes",
        "df": silver_shipment_routes_df,
        "key_columns": ["route_id", "supplier_id"],
        "record_id_column": "shipment_route_record_id"
    },
    {
        "bronze_table": "bronze_product_crosswalk",
        "silver_table": "silver_product_crosswalk",
        "df": silver_product_crosswalk_df,
        "key_columns": ["canonical_sku_id", "source_system", "source_product_id"],
        "record_id_column": "product_crosswalk_record_id"
    }
]


for item in bronze_to_silver_table_map:
    bronze_table = item["bronze_table"]
    silver_table = item["silver_table"]
    df = item["df"]
    key_columns = item["key_columns"]
    record_id_column = item["record_id_column"]

    bronze_count = spark.table(f"{BRONZE_SCHEMA}.{bronze_table}").count()
    silver_count = df.count()

    synthetic_validation_rows.append({
        "silver_table": silver_table,
        "validation_check": "row_count_preserved",
        "expected_value": str(bronze_count),
        "actual_value": str(silver_count),
        "status": "PASS" if bronze_count == silver_count else "FAIL",
        "issue_detail": None if bronze_count == silver_count else f"{silver_table} row count does not match {bronze_table}."
    })

    null_key_condition = None

    for key_col in key_columns:
        current_condition = F.col(key_col).isNull()

        if null_key_condition is None:
            null_key_condition = current_condition
        else:
            null_key_condition = null_key_condition | current_condition

    null_key_count = df.filter(null_key_condition).count()

    synthetic_validation_rows.append({
        "silver_table": silver_table,
        "validation_check": "business_keys_not_null",
        "expected_value": "0 null key rows",
        "actual_value": str(null_key_count),
        "status": "PASS" if null_key_count == 0 else "FAIL",
        "issue_detail": None if null_key_count == 0 else f"Null key values found in {silver_table}."
    })

    null_record_id_count = df.filter(F.col(record_id_column).isNull()).count()

    synthetic_validation_rows.append({
        "silver_table": silver_table,
        "validation_check": "record_id_not_null",
        "expected_value": "0 null record ids",
        "actual_value": str(null_record_id_count),
        "status": "PASS" if null_record_id_count == 0 else "FAIL",
        "issue_detail": None if null_record_id_count == 0 else f"Null {record_id_column} values found."
    })

    missing_lineage_count = (
        df
        .filter(
            F.col("synthetic_batch_id").isNull()
            | F.col("is_synthetic").isNull()
            | F.col("silver_transform_run_id").isNull()
        )
        .count()
    )

    synthetic_validation_rows.append({
        "silver_table": silver_table,
        "validation_check": "lineage_columns_not_null",
        "expected_value": "0 missing lineage rows",
        "actual_value": str(missing_lineage_count),
        "status": "PASS" if missing_lineage_count == 0 else "FAIL",
        "issue_detail": None if missing_lineage_count == 0 else f"Missing lineage metadata found in {silver_table}."
    })


# Relationship checks across supplier/internal model

supplier_count = silver_suppliers_df.select("supplier_id").distinct().count()

sku_map_missing_supplier_count = (
    silver_supplier_sku_map_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

synthetic_validation_rows.append({
    "silver_table": "silver_supplier_sku_map",
    "validation_check": "supplier_id_exists_in_suppliers",
    "expected_value": "0 orphan supplier ids",
    "actual_value": str(sku_map_missing_supplier_count),
    "status": "PASS" if sku_map_missing_supplier_count == 0 else "FAIL",
    "issue_detail": None if sku_map_missing_supplier_count == 0 else "supplier_id values found in SKU map but missing from suppliers."
})

po_missing_supplier_count = (
    silver_purchase_orders_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

synthetic_validation_rows.append({
    "silver_table": "silver_purchase_orders",
    "validation_check": "po_supplier_id_exists_in_suppliers",
    "expected_value": "0 orphan supplier ids",
    "actual_value": str(po_missing_supplier_count),
    "status": "PASS" if po_missing_supplier_count == 0 else "FAIL",
    "issue_detail": None if po_missing_supplier_count == 0 else "supplier_id values found in purchase orders but missing from suppliers."
})

route_missing_supplier_count = (
    silver_shipment_routes_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

synthetic_validation_rows.append({
    "silver_table": "silver_shipment_routes",
    "validation_check": "route_supplier_id_exists_in_suppliers",
    "expected_value": "0 orphan supplier ids",
    "actual_value": str(route_missing_supplier_count),
    "status": "PASS" if route_missing_supplier_count == 0 else "FAIL",
    "issue_detail": None if route_missing_supplier_count == 0 else "supplier_id values found in shipment routes but missing from suppliers."
})


# Numeric sanity checks

invalid_dependency_count = (
    silver_supplier_sku_map_df
    .filter((F.col("dependency_percent") < 0) | (F.col("dependency_percent") > 100))
    .count()
)

synthetic_validation_rows.append({
    "silver_table": "silver_supplier_sku_map",
    "validation_check": "dependency_percent_between_0_and_100",
    "expected_value": "0 invalid dependency rows",
    "actual_value": str(invalid_dependency_count),
    "status": "PASS" if invalid_dependency_count == 0 else "FAIL",
    "issue_detail": None if invalid_dependency_count == 0 else "dependency_percent outside 0 to 100."
})

invalid_crosswalk_confidence_count = (
    silver_product_crosswalk_df
    .filter((F.col("confidence") < 0) | (F.col("confidence") > 1))
    .count()
)

synthetic_validation_rows.append({
    "silver_table": "silver_product_crosswalk",
    "validation_check": "crosswalk_confidence_between_0_and_1",
    "expected_value": "0 invalid confidence rows",
    "actual_value": str(invalid_crosswalk_confidence_count),
    "status": "PASS" if invalid_crosswalk_confidence_count == 0 else "FAIL",
    "issue_detail": None if invalid_crosswalk_confidence_count == 0 else "confidence outside 0 to 1."
})


synthetic_validation_schema = T.StructType([
    T.StructField("silver_table", T.StringType(), True),
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

synthetic_validation_df = spark.createDataFrame(
    synthetic_validation_rows,
    schema=synthetic_validation_schema
)

display(synthetic_validation_df.orderBy("status", "silver_table", "validation_check"))

fail_count = synthetic_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver synthetic/internal validation failures:", fail_count)

In [0]:
# Notebook 12, Chunk 5: Write all Silver synthetic/internal tables

if fail_count == 0:
    for table_name, df in silver_internal_dfs.items():
        target_table = f"{SILVER_SCHEMA}.{table_name}"

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        print(f"✅ Wrote {target_table}: {df.count()} rows")

    print("✅ All Silver synthetic/internal tables written successfully.")
else:
    raise ValueError("Silver synthetic/internal validation failed. Fix issues before writing.")

In [0]:
# Notebook 12, Chunk 6: Read-back verification

expected_silver_row_counts = {
    "silver_suppliers": 48,
    "silver_supplier_aliases": 133,
    "silver_supplier_sku_map": 355,
    "silver_alternate_suppliers": 155,
    "silver_supplier_scorecards": 576,
    "silver_purchase_orders": 1707,
    "silver_shipment_routes": 97,
    "silver_product_crosswalk": 3187
}

readback_rows = []

for table_name, expected_count in expected_silver_row_counts.items():
    full_table_name = f"{SILVER_SCHEMA}.{table_name}"
    actual_count = spark.table(full_table_name).count()

    readback_rows.append({
        "silver_table": table_name,
        "expected_row_count": expected_count,
        "actual_row_count": actual_count,
        "status": "PASS" if actual_count == expected_count else "FAIL"
    })

readback_schema = T.StructType([
    T.StructField("silver_table", T.StringType(), True),
    T.StructField("expected_row_count", T.LongType(), True),
    T.StructField("actual_row_count", T.LongType(), True),
    T.StructField("status", T.StringType(), True)
])

readback_df = spark.createDataFrame(readback_rows, schema=readback_schema)

display(readback_df.orderBy("status", "silver_table"))

readback_fail_count = readback_df.filter(F.col("status") == "FAIL").count()

print("Silver synthetic/internal read-back failures:", readback_fail_count)

if readback_fail_count == 0:
    print("✅ Read-back check passed: All Silver synthetic/internal tables exist and row counts are correct.")
else:
    print("❌ Read-back check failed: Some Silver synthetic/internal tables have incorrect row counts.")

In [0]:
# Notebook 12, Chunk 7: Save validation results

synthetic_validation_results_df = (
    synthetic_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit("multiple_bronze_synthetic_internal_tables"))
    .withColumn("target_table", F.col("silver_table"))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    synthetic_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("target_table", "status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("target_table", "status")
)

print("✅ Notebook 12 PASSED: Silver synthetic/internal supplier tables created and validated.")
print("Next notebook: 13_silver_external_risk_events")